# 🛡️ Forum Toxicity Classifier — BERT Training

**Pipeline:**
1. Install & import dependencies
2. Imports & config
3. Load Jigsaw dataset (CSV)
4. Handle class imbalance & split
5. Tokenize — PyTorch Dataset
6. Build BERT classifier
7. Train
8. Training curves
9. Evaluate on test set
10. Threshold tuning
11. Export final model
12. Quick inference test
13. Flask API server
14. Symfony PHP integration
15. Download model (Colab)

## 1. Install Dependencies

In [ ]:
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

!pip install transformers datasets scikit-learn flask flask-cors matplotlib seaborn --quiet

## 2. Imports & Config

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_score,
    recall_score,
    f1_score,
)

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# ── Config ───────────────────────────────────────────────────────────────────
BERT_MODEL   = 'bert-base-uncased'
MAX_LEN      = 128
BATCH_SIZE   = 32
EPOCHS       = 3
LR           = 2e-5
WARMUP_RATIO = 0.1
THRESHOLD    = 0.5
EXPORT_DIR   = '/content/forum_toxicity_bert'
DATASET_SIZE = 50_000
SEED         = 42

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'Device      : {DEVICE}')   # should print: cuda
print(f'BERT model  : {BERT_MODEL}')
print(f'Export dir  : {EXPORT_DIR}')
print(f'Dataset rows: {DATASET_SIZE:,}')

## 3. Load Dataset

Upload `train.csv` (Jigsaw toxic comments) to Colab using the file sidebar on the left, then run this cell.

In [ ]:
df = pd.read_csv('/content/train.csv').head(DATASET_SIZE)

df = df[['comment_text', 'toxic']].rename(columns={'comment_text': 'text', 'toxic': 'label'})
df['label'] = df['label'].astype(int)
df['text']  = df['text'].astype(str).str.strip()
df          = df[df['text'].str.len() > 5].reset_index(drop=True)

print(f'Total rows : {len(df):,}')
print(f'Toxic (1)  : {df.label.sum():,}  ({df.label.mean()*100:.1f}%)')
print(f'Clean (0)  : {(df.label==0).sum():,}  ({(1-df.label.mean())*100:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['label'].value_counts().plot.bar(ax=axes[0], color=['#4a7cdc','#e05252'], title='Label distribution', rot=0)
axes[0].set_xticklabels(['Clean', 'Toxic'])
df['text'].str.len().hist(ax=axes[1], bins=60, color='#4a7cdc', alpha=0.8)
axes[1].set_title('Comment length distribution (chars)')
plt.tight_layout()
plt.show()

## 4. Handle Class Imbalance & Split

In [ ]:
n_clean    = (df.label == 0).sum()
n_toxic    = (df.label == 1).sum()
pos_weight = torch.tensor([n_clean / n_toxic], dtype=torch.float).to(DEVICE)
print(f'Class weight for toxic class: {pos_weight.item():.2f}x')

# Train / val / test  —  80 / 10 / 10  stratified
train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=SEED)
val_df,   test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=SEED)

print(f'Train : {len(train_df):,}')
print(f'Val   : {len(val_df):,}')
print(f'Test  : {len(test_df):,}')

## 5. Tokenize — PyTorch Dataset

In [ ]:
tokenizer = BertTokenizer.from_pretrained(BERT_MODEL)

class ForumDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts.reset_index(drop=True)
        self.labels    = labels.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length     = self.max_len,
            padding        = 'max_length',
            truncation     = True,
            return_tensors = 'pt',
        )
        return {
            'input_ids'      : enc['input_ids'].squeeze(),
            'attention_mask' : enc['attention_mask'].squeeze(),
            'token_type_ids' : enc['token_type_ids'].squeeze(),
            'labels'         : torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_ds = ForumDataset(train_df['text'], train_df['label'], tokenizer, MAX_LEN)
val_ds   = ForumDataset(val_df['text'],   val_df['label'],   tokenizer, MAX_LEN)
test_ds  = ForumDataset(test_df['text'],  test_df['label'],  tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')

## 6. Build BERT Classifier

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    BERT_MODEL,
    num_labels          = 2,
    hidden_dropout_prob = 0.3,
).to(DEVICE)

optimizer    = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
loss_fn      = torch.nn.CrossEntropyLoss(
                   weight=torch.tensor([1.0, pos_weight.item()]).to(DEVICE)
               )

print(f'Parameters   : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
print(f'Total steps  : {total_steps:,}  |  Warmup: {warmup_steps:,}')

## 7. Train

In [ ]:
import os

def evaluate(loader):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_probs, all_labels = [], [], []
    with torch.no_grad():
        for batch in loader:
            ids   = batch['input_ids'].to(DEVICE)
            mask  = batch['attention_mask'].to(DEVICE)
            ttype = batch['token_type_ids'].to(DEVICE)
            labs  = batch['labels'].to(DEVICE)
            out   = model(ids, attention_mask=mask, token_type_ids=ttype)
            loss  = loss_fn(out.logits, labs)
            probs = torch.softmax(out.logits, dim=1)[:, 1]
            preds = (probs >= THRESHOLD).long()
            total_loss += loss.item()
            correct    += (preds == labs).sum().item()
            total      += labs.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labs.cpu().numpy())
    return total_loss / len(loader), correct / total, all_preds, all_probs, all_labels


history       = {'train_loss': [], 'val_loss': [], 'val_acc': []}
best_val_loss = float('inf')
os.makedirs(EXPORT_DIR + '/best_checkpoint', exist_ok=True)

for epoch in range(EPOCHS):
    model.train()
    run_loss, correct, total = 0, 0, 0

    for step, batch in enumerate(train_loader):
        optimizer.zero_grad()
        ids   = batch['input_ids'].to(DEVICE)
        mask  = batch['attention_mask'].to(DEVICE)
        ttype = batch['token_type_ids'].to(DEVICE)
        labs  = batch['labels'].to(DEVICE)
        out   = model(ids, attention_mask=mask, token_type_ids=ttype)
        loss  = loss_fn(out.logits, labs)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        probs    = torch.softmax(out.logits, dim=1)[:, 1]
        preds    = (probs >= THRESHOLD).long()
        run_loss += loss.item()
        correct  += (preds == labs).sum().item()
        total    += labs.size(0)
        if step % 100 == 0:
            print(f'  Epoch {epoch+1}/{EPOCHS} | Step {step}/{len(train_loader)} | Loss {loss.item():.4f}')

    val_loss, val_acc, _, _, _ = evaluate(val_loader)
    avg_loss = run_loss / len(train_loader)
    history['train_loss'].append(avg_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    print(f'\nEpoch {epoch+1} — Train Loss: {avg_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}%')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        model.save_pretrained(EXPORT_DIR + '/best_checkpoint')
        tokenizer.save_pretrained(EXPORT_DIR + '/best_checkpoint')
        print(f'  ✔ Best model saved (val_loss={best_val_loss:.4f})')

print('\nTraining complete!')

## 8. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], marker='o', label='Train')
axes[0].plot(history['val_loss'],   marker='o', label='Val')
axes[0].set_title('Loss per epoch'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].plot([v*100 for v in history['val_acc']], marker='o', color='green')
axes[1].set_title('Validation accuracy (%)'); axes[1].set_xlabel('Epoch'); axes[1].set_ylim([0, 100])
plt.tight_layout()
plt.show()

## 9. Evaluate on Test Set

In [ ]:
best_model = BertForSequenceClassification.from_pretrained(EXPORT_DIR + '/best_checkpoint').to(DEVICE)
best_model.eval()

_, test_acc, test_preds, test_probs, test_labels = evaluate(test_loader)

print('=' * 50)
print(f'Test Accuracy : {test_acc*100:.2f}%')
print(f'ROC-AUC       : {roc_auc_score(test_labels, test_probs):.4f}')
print('=' * 50)
print(classification_report(test_labels, test_preds, target_names=['Clean', 'Toxic']))

In [ ]:
cm = confusion_matrix(test_labels, test_preds)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Clean','Toxic'], yticklabels=['Clean','Toxic'])
axes[0].set_title('Confusion Matrix'); axes[0].set_ylabel('Actual'); axes[0].set_xlabel('Predicted')
fpr, tpr, _ = roc_curve(test_labels, test_probs)
auc_score   = roc_auc_score(test_labels, test_probs)
axes[1].plot(fpr, tpr, color='#4a7cdc', lw=2, label=f'AUC = {auc_score:.3f}')
axes[1].plot([0,1],[0,1],'--',color='gray')
axes[1].set_title('ROC Curve'); axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR'); axes[1].legend()
plt.tight_layout()
plt.show()

## 10. Threshold Tuning

In [ ]:
thresholds = np.arange(0.1, 0.95, 0.05)
rows = []
for t in thresholds:
    p = (np.array(test_probs) >= t).astype(int)
    rows.append({
        'threshold': round(t, 2),
        'precision': round(precision_score(test_labels, p, zero_division=0), 3),
        'recall'   : round(recall_score(test_labels, p, zero_division=0), 3),
        'f1'       : round(f1_score(test_labels, p, zero_division=0), 3),
    })

thresh_df   = pd.DataFrame(rows)
best_thresh = thresh_df.loc[thresh_df['f1'].idxmax()]
THRESHOLD   = float(best_thresh['threshold'])

print(thresh_df.to_string(index=False))
print(f'\n→ Best threshold by F1: {THRESHOLD}  (F1={best_thresh["f1"]})')

plt.figure(figsize=(9, 4))
plt.plot(thresh_df['threshold'], thresh_df['precision'], marker='o', label='Precision')
plt.plot(thresh_df['threshold'], thresh_df['recall'],    marker='o', label='Recall')
plt.plot(thresh_df['threshold'], thresh_df['f1'],        marker='o', label='F1', linestyle='--')
plt.axvline(x=THRESHOLD, color='red', linestyle=':', label=f'Selected ({THRESHOLD})')
plt.xlabel('Threshold'); plt.title('Precision / Recall / F1 vs Threshold'); plt.legend()
plt.tight_layout()
plt.show()

## 11. Export Final Model

In [ ]:
import os

os.makedirs(EXPORT_DIR, exist_ok=True)
best_model.save_pretrained(EXPORT_DIR)
tokenizer.save_pretrained(EXPORT_DIR)

config = {
    'model_dir'  : EXPORT_DIR,
    'threshold'  : THRESHOLD,
    'max_len'    : MAX_LEN,
    'labels'     : {'0': 'clean', '1': 'toxic'},
    'bert_model' : BERT_MODEL,
}
with open(os.path.join(EXPORT_DIR, 'config.json'), 'w') as f:
    json.dump(config, f, indent=2)

print('✔ Model exported to:', EXPORT_DIR)
for fname in os.listdir(EXPORT_DIR):
    size = os.path.getsize(os.path.join(EXPORT_DIR, fname))
    print(f'  {fname:<40} {size/1e6:.1f} MB' if size > 100_000 else f'  {fname}')

## 12. Quick Inference Test

In [ ]:
def predict_text(text, model, tokenizer, threshold=THRESHOLD):
    model.eval()
    enc  = tokenizer(
        text,
        max_length     = MAX_LEN,
        padding        = 'max_length',
        truncation     = True,
        return_tensors = 'pt',
    ).to(DEVICE)
    with torch.no_grad():
        prob = torch.softmax(best_model(**enc).logits, dim=1)[0, 1].item()
    return {
        'text'       : text[:80],
        'toxic_prob' : round(prob, 4),
        'label'      : 'toxic' if prob >= threshold else 'clean',
    }

test_comments = [
    'Great post! Thanks for sharing this.',
    'You are completely wrong and frankly an idiot.',
    'I disagree but I see where you are coming from.',
    'This thread is garbage and so are you.',
    'Can anyone explain how to configure this setting?',
    'I will find you and make you regret this.',
]

print(f'{"Comment":<50} {"Prob":>6}  Label')
print('-' * 68)
for c in test_comments:
    r    = predict_text(c, best_model, tokenizer)
    flag = '🚨' if r['label'] == 'toxic' else '✅'
    print(f'{c[:49]:<50} {r["toxic_prob"]:>6.3f}  {flag} {r["label"]}')

## 13. Write Flask API Server

This writes `api_server.py` which you run on your machine alongside Symfony:
```bash
python api_server.py
# → http://127.0.0.1:5000
```

**Endpoints:**
- `GET  /health`         — health check
- `POST /predict`        — `{"text": "..."}` → `{"is_toxic": bool, "toxic_probability": float, "label": str}`
- `POST /predict/batch`  — `{"texts": [...]}` → `{"results": [...]}`

In [ ]:
api_code = '''
import json
import torch
from flask import Flask, request, jsonify
from flask_cors import CORS
from transformers import BertTokenizer, BertForSequenceClassification

MODEL_DIR = "./forum_toxicity_bert"

with open(f"{MODEL_DIR}/config.json") as f:
    cfg = json.load(f)

THRESHOLD = cfg["threshold"]
MAX_LEN   = cfg["max_len"]
DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Loading model...")
tokenizer = BertTokenizer.from_pretrained(MODEL_DIR)
model     = BertForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE)
model.eval()
print(f"Ready on {DEVICE}. Threshold={THRESHOLD}")

app = Flask(__name__)
CORS(app)

def infer(text):
    enc  = tokenizer(
        text,
        max_length     = MAX_LEN,
        padding        = "max_length",
        truncation     = True,
        return_tensors = "pt",
    ).to(DEVICE)
    with torch.no_grad():
        prob = torch.softmax(model(**enc).logits, dim=1)[0, 1].item()
    return {
        "toxic_probability" : round(prob, 4),
        "label"             : "toxic" if prob >= THRESHOLD else "clean",
        "is_toxic"          : bool(prob >= THRESHOLD),
        "threshold"         : THRESHOLD,
    }

@app.route("/health")
def health():
    return jsonify({"status": "ok", "device": str(DEVICE)})

@app.route("/predict", methods=["POST"])
def predict():
    data = request.get_json(force=True)
    if not data or "text" not in data:
        return jsonify({"error": "Missing `text` field"}), 400
    return jsonify(infer(str(data["text"]).strip()))

@app.route("/predict/batch", methods=["POST"])
def predict_batch():
    data  = request.get_json(force=True)
    texts = data.get("texts", [])
    if not isinstance(texts, list):
        return jsonify({"error": 'Send {"texts": [str, ...]}'}), 400
    return jsonify({"results": [infer(t) for t in texts[:50]]})

if __name__ == "__main__":
    app.run(host="127.0.0.1", port=5000, debug=False)
'''

with open('/content/api_server.py', 'w') as f:
    f.write(api_code.strip())

print('✔ api_server.py written to /content/api_server.py')

## 14. Symfony PHP Integration

### `src/Service/ToxicityService.php`

```php
<?php
namespace App\Service;

use Symfony\Contracts\HttpClient\HttpClientInterface;

class ToxicityService
{
    private const API_URL = 'http://127.0.0.1:5000';

    public function __construct(private HttpClientInterface $http) {}

    public function analyse(string $text): array
    {
        return $this->http
            ->request('POST', self::API_URL . '/predict', [
                'json'    => ['text' => $text],
                'timeout' => 5.0,
            ])
            ->toArray();
    }

    public function analyseBatch(array $texts): array
    {
        return $this->http
            ->request('POST', self::API_URL . '/predict/batch', [
                'json'    => ['texts' => $texts],
                'timeout' => 15.0,
            ])
            ->toArray()['results'];
    }

    public function isToxic(string $text): bool
    {
        return $this->analyse($text)['is_toxic'] ?? false;
    }
}
```

---

### `src/EventListener/CommentModerationListener.php`

```php
<?php
namespace App\EventListener;

use App\Entity\Comment;
use App\Service\ToxicityService;
use Doctrine\Bundle\DoctrineBundle\Attribute\AsEntityListener;
use Doctrine\ORM\Events;
use Doctrine\ORM\Event\PrePersistEventArgs;
use Doctrine\ORM\Event\PreUpdateEventArgs;

#[AsEntityListener(event: Events::prePersist, entity: Comment::class)]
#[AsEntityListener(event: Events::preUpdate,  entity: Comment::class)]
class CommentModerationListener
{
    public function __construct(private ToxicityService $toxicity) {}

    public function prePersist(Comment $comment, PrePersistEventArgs $args): void
    {
        $this->moderate($comment);
    }

    public function preUpdate(Comment $comment, PreUpdateEventArgs $args): void
    {
        if ($args->hasChangedField('content')) {
            $this->moderate($comment);
        }
    }

    private function moderate(Comment $comment): void
    {
        $result = $this->toxicity->analyse($comment->getContent());

        $comment->setToxicityScore($result['toxic_probability']);
        $comment->setIsFlagged($result['is_toxic']);

        // Auto-hide comments with very high toxicity
        if ($result['toxic_probability'] >= 0.85) {
            $comment->setStatus('hidden');
        }
    }
}
```

---

### Add fields to your Comment entity

```php
#[ORM\Column(type: 'float', nullable: true)]
private ?float $toxicityScore = null;

#[ORM\Column(type: 'boolean', options: ['default' => false])]
private bool $isFlagged = false;

#[ORM\Column(type: 'string', length: 20, options: ['default' => 'visible'])]
private string $status = 'visible';  // visible | hidden | deleted
```

Then run:
```bash
php bin/console doctrine:migrations:diff
php bin/console doctrine:migrations:migrate
```

## 15. Download Model from Colab

Run this last cell to zip everything and download it directly to your computer.

In [ ]:
import shutil
from google.colab import files

# Copy api_server.py into the model folder so everything is in one zip
shutil.copy('/content/api_server.py', EXPORT_DIR + '/api_server.py')

# Zip the folder
shutil.make_archive('/content/forum_toxicity_bert', 'zip', '/content', 'forum_toxicity_bert')

print('✔ Zipped. Starting download...')
files.download('/content/forum_toxicity_bert.zip')